In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score

df = pd.read_csv("abstention_mistral_final.csv")

df["final_answer"] = df["final_answer"].astype(str).str.lower().str.strip()
df["pred_final_answer"] = df["pred_final_answer"].astype(str).str.lower().str.strip()

df["reasoning"] = df["reasoning"].fillna("")
df["pred_reasoning"] = df["pred_reasoning"].fillna("")

accuracy = accuracy_score(df["final_answer"], df["pred_final_answer"])

smooth = SmoothingFunction().method1

rouge_scorer_obj = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def compute_token_f1(ref, pred):
    ref_tokens = ref.split()
    pred_tokens = pred.split()

    common = set(ref_tokens) & set(pred_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens) if pred_tokens else 0.0
    recall = len(common) / len(ref_tokens) if ref_tokens else 0.0

    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)

def compute_binary_correct(true, pred):
    true = str(true).lower().strip()
    pred = str(pred).lower().strip()

    if true == "idk" and pred == "idk":
        return 1

    if (true == "idk" and pred in ["a", "b", "c", "d"]) or \
       (pred == "idk" and true in ["a", "b", "c", "d"]):
        return 0

    if true in ["a", "b", "c", "d"] and pred in ["a", "b", "c", "d"]:
        return 1

    return 0


df["binary_correct"] = [
    compute_binary_correct(t, p)
    for t, p in zip(df["final_answer"], df["pred_final_answer"])
]

binary_accuracy = np.mean(df["binary_correct"])

bleu_list = []
rouge1_list = []
rouge2_list = []
rougeL_list = []
token_f1_list = []

for ref, pred in zip(df["reasoning"], df["pred_reasoning"]):

    ref_tokens = ref.split()
    pred_tokens = pred.split()

    if len(pred_tokens) == 0:
        bleu = 0.0
    else:
        bleu = sentence_bleu(
            [ref_tokens],
            pred_tokens,
            smoothing_function=smooth
        )

    bleu_list.append(bleu)

    scores = rouge_scorer_obj.score(ref, pred)
    rouge1_list.append(scores["rouge1"].fmeasure)
    rouge2_list.append(scores["rouge2"].fmeasure)
    rougeL_list.append(scores["rougeL"].fmeasure)

    token_f1_list.append(compute_token_f1(ref, pred))


P, R, F1 = bert_score(
    df["pred_reasoning"].tolist(),
    df["reasoning"].tolist(),
    lang="en",
    verbose=True
)

df["bleu"] = bleu_list
df["rouge1"] = rouge1_list
df["rouge2"] = rouge2_list
df["rougeL"] = rougeL_list
df["token_f1"] = token_f1_list
df["bertscore_precision"] = P.tolist()
df["bertscore_recall"] = R.tolist()
df["bertscore_f1"] = F1.tolist()

df["correct"] = (df["final_answer"] == df["pred_final_answer"]).astype(int)

avg_bleu = np.mean(bleu_list)
avg_rouge1 = np.mean(rouge1_list)
avg_rouge2 = np.mean(rouge2_list)
avg_rougeL = np.mean(rougeL_list)
avg_token_f1 = np.mean(token_f1_list)
avg_bertscore_f1 = F1.mean().item()


print("\n===== OVERALL METRICS =====")
print(f"Accuracy       : {accuracy:.4f}")
print(f"Binary Accuracy: {binary_accuracy:.4f}")
print(f"BLEU           : {avg_bleu:.4f}")
print(f"ROUGE-1        : {avg_rouge1:.4f}")
print(f"ROUGE-2        : {avg_rouge2:.4f}")
print(f"ROUGE-L        : {avg_rougeL:.4f}")
print(f"BERTScore F1   : {avg_bertscore_f1:.4f}")
print(f"Token F1       : {avg_token_f1:.4f}")

df.to_csv("evaluation_mistral.csv", index=False)

print("\nSaved detailed metrics to evaluation_mistral.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/88 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/56 [00:00<?, ?it/s]

done in 18.06 seconds, 194.93 sentences/sec

===== OVERALL METRICS =====
Accuracy       : 0.2945
Binary Accuracy: 0.7211
BLEU           : 0.0189
ROUGE-1        : 0.1839
ROUGE-2        : 0.0617
ROUGE-L        : 0.1508
BERTScore F1   : 0.6039
Token F1       : 0.1158

Saved detailed metrics to evaluation_mistral.csv
